In [38]:
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import json
sys.path.append(os.path.dirname(os.path.abspath('.')))
from utils.load_api_keys import get_env_var
from pathlib import Path
from pprint import pprint

import glob

In [39]:
HF_TOKEN = get_env_var("HF_TOKEN")
GROQ_API_KEY = get_env_var("GROQ_API_KEY")
FILE_PATH_MAJORS = get_env_var("FILE_PATH_MAJORS")
PERSIST_DIR = get_env_var("PERSIST_DIR")
PERSIST_DIR_FAISS = get_env_var("PERSIST_DIR_FAISS")
OPENAI_API_KEY = get_env_var("OPENAI_API_KEY")
FILE_PATH_MAJORS

'../data/Majors/'

In [40]:
from langchain_community.document_loaders import JSONLoader, MergedDataLoader, WebBaseLoader
from langchain_text_splitters import RecursiveJsonSplitter, RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
#from langchain.embeddings import OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage


In [41]:
#loading the embedding model and llm model 
#embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#model = ChatGroq(model="Gemma2-9b-It", groq_api_key =  GROQ_API_KEY)
#model
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=OPENAI_API_KEY)
model = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)

In [42]:
urls = ["https://www.qatar.cmu.edu/academics-research/academics/information-systems/",
        "https://www.qatar.cmu.edu/academics-research/academics/computer-science/"]
docs = [WebBaseLoader(url).load() for url in urls]
doc_list = [ doc for sublist in docs for doc in sublist]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc_split = text_splitter.split_documents(doc_list) 
print(doc_split[:2])

vectorstore = FAISS.from_documents(doc_split, embeddings)
retriever = vectorstore.as_retriever()
retriever.invoke("What is the name of the university?")


[Document(metadata={'source': 'https://www.qatar.cmu.edu/academics-research/academics/information-systems/', 'title': 'CMU-Q Information Systems | Globally top ranked program', 'description': 'From programming to project management to creating new ventures, the field of information systems creates value by using technology to generate, process and distribute information in an effective, efficient way.', 'language': 'en-US'}, page_content='CMU-Q Information Systems | Globally top ranked program\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n\n\nYour Path to CMU\n\n\n                          Follow these steps to become a CMU Qatar student                      \n\n Explore\n\n\n\n                        Learn about academic programs                      \n\n\n                        Learn about CMU Qatar                      \n\n\n                        Learn about our student life                      \n\n\

[Document(id='49436234-5961-4938-9958-6e785ababe2f', metadata={'source': 'https://www.qatar.cmu.edu/academics-research/academics/information-systems/', 'title': 'CMU-Q Information Systems | Globally top ranked program', 'description': 'From programming to project management to creating new ventures, the field of information systems creates value by using technology to generate, process and distribute information in an effective, efficient way.', 'language': 'en-US'}, page_content='Teaching Professor, Information Systems              \n\n\n\nSelma Limam Mansar\n\n\n                Teaching Professor of Information Systems Emeritus              \n\n\n\nManoranjan Mohanty\n\n\n                Assistant Teaching Professor              \n\n\n\n\nDaniel C. Phelps\n\n\n                Associate Teaching Professor Emeritus              \n\n\n\nNui Vatanasakdakul\n\n\n                Teaching Professor, Information Systems              \n\n\n\n\n\n\n\n\n\n\n \nContact us\n\nCalendar\nCareers\nD

In [43]:
json_CSmajor_files = glob.glob(os.path.join(FILE_PATH_MAJORS,'Computer Science/','*.json'))
for f in json_CSmajor_files:
    majorfilename = Path(f).stem
json_Courses_files = glob.glob(os.path.join(r'../data/Courses/','*.json'))
for c in json_Courses_files:
    coursefilename = Path(c).stem
    

In [44]:
len(json_Courses_files)


4736

In [45]:
int(len(json_Courses_files)/4)
for i in range(0,len(json_Courses_files),int(len(json_Courses_files)/8)):
    print(i)

0
592
1184
1776
2368
2960
3552
4144


In [46]:
course_data = []
for f in json_Courses_files:
    try:
        with open(str(f), 'r', encoding='utf-8') as file:
            data = json.load(file)
            doc = Document(page_content=json.dumps(data), metadata={'source': Path(f).stem})
            course_data.append(doc)
    except json.JSONDecodeError as e:
        print(f"Invalid JSON in file {f}: {e}")
    except Exception as e:
        print(f"Error reading file {f}: {e}")



In [47]:
type(course_data[0])

langchain_core.documents.base.Document

In [48]:
'''from langchain_text_splitters import RecursiveJsonSplitter

splitter = RecursiveJsonSplitter(max_chunk_size=1000)
json_chunks = splitter.split_json(json_data=course_data, convert_lists=True)'''

'from langchain_text_splitters import RecursiveJsonSplitter\n\nsplitter = RecursiveJsonSplitter(max_chunk_size=1000)\njson_chunks = splitter.split_json(json_data=course_data, convert_lists=True)'

In [49]:
'''for chunk in json_chunks[-10:]:
    for p,m in chunk.items():
        print(p,m)'''

'for chunk in json_chunks[-10:]:\n    for p,m in chunk.items():\n        print(p,m)'

In [50]:
def metadata_func(record: dict, metadata: dict) -> dict:
    if "source" in metadata:
        metadata["source"] = Path(metadata["source"]).stem
    metadata["department"] = "CS"  
    metadata["category"] = "CS Major"
    return metadata

In [51]:
def metadata_func_Courses(record: dict, metadata: dict) -> dict:
    if "source" in metadata:
        metadata["source"] = Path(metadata["source"]).stem
    metadata["departmentid"] = Path(metadata["source"]).stem.split('-')[0]  
    metadata['category'] = 'Courses'
    return metadata

In [52]:
CSMajor_docs =[]
for file in json_CSmajor_files:
    loader = JSONLoader(file_path=file,
                        jq_schema=".[]",
                        text_content=False,
                        metadata_func=metadata_func)
    raw_docs = loader.load()
    CSMajor_docs.extend(raw_docs)

print("length of docs:",len(CSMajor_docs))
print(CSMajor_docs[0].metadata,"\n",CSMajor_docs[0].page_content)


length of docs: 113
{'source': 'CMUQ_CS_Advising_Document', 'seq_num': 1, 'department': 'CS', 'category': 'CS Major'} 
 {"Subheading": "Advisor Role and Resources", "Content": "Every student at CMUQ has an academic advisor to help them ensure they are making progress towards fulfilling their major requirements. The official advisor is typically a faculty in their home department, and is listed in the students' S3 and Stellic. In the Computer Science department, there is one advisor for first year students, and one advisor for each class that advises them from sophomore year until graduation. The academic advisor is part of the overall students' support system at CMUQ, and the list below summarizes the expected responsibilities of advisors. This is not an exhaustive list.\u25cf Help with planning courses to ensure progress towards graduation. \u25cf Adding exceptions for courses in Stellic once they are approved by the appropriate department or faculty (e.g. counting senior theses for c

In [53]:
#all_courses
Courses_docs = []

for file in json_Courses_files:
    loader = JSONLoader(file_path=file,
                            jq_schema="""
                                    def clean:
                                        
                                        if type == "object" then
                                            with_entries(select(.value != null
                                                and (( ( (.value | type) != "object" and (.value | type) != "array") and .value != "" )
                                                or ( ((.value | type) == "object" or (.value | type) == "array") and ((.value | clean | length) > 0) ))))
                                        elif type == "array" then
                                            map(clean) | map(select(. != null and . != "" and ((type != "object" and type != "array") or (length > 0))))
                                        else
                                            .
                                        end;
                                    clean
                                    """,
                            text_content=False,
                            metadata_func=metadata_func_Courses)
    raw_docs = loader.load()
    Courses_docs.extend(raw_docs)
print("length of docs:",len(Courses_docs))
print(Courses_docs[0].metadata,"\n",Courses_docs[0].page_content)

length of docs: 4736
{'source': '02-201', 'seq_num': 1, 'departmentid': '02', 'category': 'Courses'} 
 {"code": "02-201", "name": "Programming for Scientists", "base_name": "Programming for Scientists", "units": 10, "min_units": 10, "max_units": 10, "short_name": "PRGRMMING SCIENTISTS", "is_topic": false, "offered_in_campuses": [1], "offerings": [{"campus_id": 1, "semesters": [{"semester": 2, "year": 2016}, {"semester": 1, "year": 2016}, {"semester": 2, "year": 2017}, {"semester": 2, "year": 2018}, {"semester": 2, "year": 2019}], "sub_semesters": []}], "long_desc": "Provides a practical introduction to programming for students with little or no prior programming experience who are interested in science. Fundamental scientific algorithms will be introduced, and extensive programming assignments will be based on analytical tasks that might be faced by scientists, such as parsing,  simulation, and optimization.  Principles of good software engineering will also be stressed. The course wil

In [ ]:
'''Courses_docs = []

for file in json_Courses_files:
    #if(Path(file).stem.split('-')[0]=='15'):
    loader = JSONLoader(file_path=file,
                            jq_schema=".",
                            text_content=False,
                            metadata_func=metadata_func_Courses)
    raw_docs = loader.load()
    Courses_docs.extend(raw_docs)
print("length of docs:",len(Courses_docs))
print(Courses_docs[0].metadata,"\n",Courses_docs[0].page_content)'''

length of docs: 4736
{'source': '02-201', 'seq_num': 1, 'departmentid': '02', 'category': 'Courses'} 
 {"code": "02-201", "name": "Programming for Scientists", "base_name": "Programming for Scientists", "units": 10, "min_units": 10, "max_units": 10, "short_name": "PRGRMMING SCIENTISTS", "is_topic": false, "topic": null, "prereqs": {"text": "", "req_obj": null, "raw_pre_req": ""}, "offered_in_campuses": [1], "offerings": [{"campus_id": 1, "semesters": [{"semester": 2, "year": 2016}, {"semester": 1, "year": 2016}, {"semester": 2, "year": 2017}, {"semester": 2, "year": 2018}, {"semester": 2, "year": 2019}], "sub_semesters": []}], "co_reqs": [], "anti_reqs": [], "equiv": [], "long_desc": "Provides a practical introduction to programming for students with little or no prior programming experience who are interested in science. Fundamental scientific algorithms will be introduced, and extensive programming assignments will be based on analytical tasks that might be faced by scientists, such 

In [18]:

#jsonsplitter = RecursiveJsonSplitter(max_chunk_size=1000)
#jsonsplitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=500)
#json_chunks = jsonsplitter.create_documents(docs,convert_lists=True)
#json_chunks = jsonsplitter.split_documents(docs)
# '''


In [19]:
jsonsplitter = RecursiveJsonSplitter(max_chunk_size=1000)
# Create chunks with metadata
ISMajor_json_chunks = []
for doc in CSMajor_docs:
    if isinstance(doc.page_content, str):
        content = json.loads(doc.page_content)
    else:
        content = doc.page_content
        
    chunks = jsonsplitter.split_text(content, convert_lists=True)
    
    for chunk in chunks:
        ISMajor_json_chunks.append(Document(
            page_content=chunk,
            metadata=doc.metadata
        ))
for jc in ISMajor_json_chunks[:5]:
    print(jc.metadata)
    print(jc.page_content)
    print("-"*100)

{'source': 'CMUQ_CS_Advising_Document', 'seq_num': 1, 'department': 'CS', 'category': 'CS Major'}
{"Subheading": "Advisor Role and Resources", "Content": "Every student at CMUQ has an academic advisor to help them ensure they are making progress towards fulfilling their major requirements. The official advisor is typically a faculty in their home department, and is listed in the students' S3 and Stellic. In the Computer Science department, there is one advisor for first year students, and one advisor for each class that advises them from sophomore year until graduation. The academic advisor is part of the overall students' support system at CMUQ, and the list below summarizes the expected responsibilities of advisors. This is not an exhaustive list.\u25cf Help with planning courses to ensure progress towards graduation. \u25cf Adding exceptions for courses in Stellic once they are approved by the appropriate department or faculty (e.g. counting senior theses for concentrations, placeme

In [54]:
splitter = RecursiveJsonSplitter(max_chunk_size=1000)
Courses_json_chunks = []

for doc in Courses_docs:  # or all docs
    if isinstance(doc.page_content, str):
        content = json.loads(doc.page_content)  # convert JSON string to dict

    chunks = splitter.split_json(content)
    print(chunks)
    for c in chunks:
        print(len(chunks),doc.metadata['source'])
        Courses_json_chunks.append(Document(
            page_content = json.dumps(c, ensure_ascii=False),
            metadata = doc.metadata
        ))

[{'code': '02-201', 'name': 'Programming for Scientists', 'base_name': 'Programming for Scientists', 'units': 10, 'min_units': 10, 'max_units': 10, 'short_name': 'PRGRMMING SCIENTISTS', 'is_topic': False, 'offered_in_campuses': [1], 'offerings': [{'campus_id': 1, 'semesters': [{'semester': 2, 'year': 2016}, {'semester': 1, 'year': 2016}, {'semester': 2, 'year': 2017}, {'semester': 2, 'year': 2018}, {'semester': 2, 'year': 2019}], 'sub_semesters': []}], 'long_desc': 'Provides a practical introduction to programming for students with little or no prior programming experience who are interested in science. Fundamental scientific algorithms will be introduced, and extensive programming assignments will be based on analytical tasks that might be faced by scientists, such as parsing,  simulation, and optimization.  Principles of good software engineering will also be stressed. The course will introduce students to the Go programming language, an industrysupported, modern programming language

In [55]:
from tqdm import tqdm
def store_documents_in_faiss(
    documents: list[Document],
    persist_dir: str = "",
    embedding_model=None,
    batch_size: int = 100,
):
    """
    Store a list of LangChain Document objects in ChromaDB with batching.

    Args:
        documents : List of documents to embed and store.
        persist_dir : Directory to persist ChromaDB.
        embedding_model: Embedding model .
        batch_size : Number of documents to process per batch.
    """
    if embedding_model is None:
        embedding_model = embeddings
    first_batch = documents[:batch_size]
    remaining = documents[batch_size:]
    #initialize with first batch
    vectordb = FAISS.from_documents(first_batch,
        embedding_model
    )
    #add remaining batches
    for i in tqdm(range(0, len(remaining), batch_size), desc="Storing documents in Faiss"):
        batch = remaining[i:i + batch_size]
        vectordb.add_documents(batch)

    # Persist to disk
    vectordb.save_local(persist_dir)

    print(f"Stored {len(documents)} documents in '{persist_dir}'.")

    return vectordb

In [56]:
#ISmajor_db=store_documents_in_faiss(ISMajor_json_chunks,persist_dir=PERSIST_DIR_FAISS+"CS/MajorData")
courses_db=store_documents_in_faiss(Courses_json_chunks,persist_dir=PERSIST_DIR_FAISS+"Courses")
#retriever_major = ISmajor_db.as_retriever()
#retriever_courses = courses_db.as_retriever()

Storing documents in Faiss: 100%|██████████| 92/92 [03:30<00:00,  2.29s/it]


Stored 9232 documents in '../faiss_db/Courses'.


In [27]:
db_major = FAISS.load_local(PERSIST_DIR_FAISS + 'CS/MajorData', embeddings, allow_dangerous_deserialization=True)
CSMajor_retriever = db_major.as_retriever()
db_course = FAISS.load_local(PERSIST_DIR_FAISS + 'Courses', embeddings, allow_dangerous_deserialization=True)
Courses_retriever = db_course.as_retriever(search_kwargs={"k":10,})

In [57]:
Courses_retriever = courses_db.as_retriever(search_kwargs={"k":10,})

In [ ]:
docs = Courses_retriever.invoke(
    "67-373",
    top_k=10,
    filters={
        'source':'67-373',
        'departmentid':'67'
    }    
)
print(docs)
for doc in docs:
    content_dict = json.loads(doc.page_content)
    for j,k in content_dict.items():
        if j=="code":
            print(k)

[Document(id='3c8dab12-8c84-42a0-82cf-fb3d90923c52', metadata={'source': '73-357', 'seq_num': 1, 'departmentid': '73', 'category': 'Courses'}, page_content='{"code": "73-357", "name": "73-357", "base_name": "73-357", "units": 0, "is_topic": false, "is_repeatable": false, "is_req_repeatable": false, "success": true}'), Document(id='fdd337f5-f503-4e0b-957a-b8ad27b54f37', metadata={'source': '67-315', 'seq_num': 1, 'departmentid': '67', 'category': 'Courses'}, page_content='{"code": "67-315", "name": "A Web For Everyone", "base_name": "A Web For Everyone", "units": 9, "min_units": 9, "max_units": 9, "short_name": "A WEB FOR EVERYONE", "is_topic": false, "prereqs": {"text": "(67-272 [] at least D) or (67-240 [] at least D)", "req_obj": {"id": 7910420, "screen_name": "9sneg9aafqzq4j2", "original_min_units": null, "min_units": null, "is_shared": false, "is_uni_req": false, "is_concentration": false, "default_concentration": false, "choices": [{"id": 7910421, "screen_name": "1y66qmhr14jwnhh",

In [35]:
# Retrieve a broader set of documents
#Courses_retriever = db_course.as_retriever(search_kwargs={"k":10,})
'''docs = Courses_retriever.invoke("67-373",
                                top_k=10,
                                filters={
                                    'source':'67-373',
                                    'departmentid':'67'
                                }  )'''
print(docs)
# Manually filter based on metadata


[Document(id='c806da1f-174d-4527-8eb1-4933054c000d', metadata={'source': '67-315', 'seq_num': 1, 'departmentid': '67', 'category': 'Courses'}, page_content='{"code": "67-315", "name": "A Web For Everyone", "base_name": "A Web For Everyone", "units": 9, "min_units": 9, "max_units": 9, "short_name": "A WEB FOR EVERYONE", "is_topic": false, "topic": null, "prereqs": {"text": "(67-272 [] at least D) or (67-240 [] at least D)", "req_obj": {"id": 7910420, "screen_name": "9sneg9aafqzq4j2", "original_min_units": null, "min_units": null, "is_shared": false, "is_uni_req": false, "is_concentration": false, "default_concentration": false, "choices": [{"id": 7910421, "screen_name": "1y66qmhr14jwnhh", "original_min_units": null, "min_units": null, "is_shared": false, "is_uni_req": false, "is_concentration": false, "default_concentration": false, "choices": [{"id": 7076, "screen_name": "67-272", "constraints": [{"type": "course", "type_string": "", "data": {"course": {"code": "67-272", "id": 7075, "n

In [56]:
docs

[Document(id='92c20180-c019-4152-b1e3-e81445fe47aa', metadata={'source': '67-373', 'seq_num': 1, 'departmentid': '67', 'category': 'Courses'}, page_content='{"long_desc": "Information Systems IS Consulting Project is a junior level teambased course that focuses on working as a team to build a solution to meet the needs of a client. With your teammates, you will work with an actual client to design, build, and deliver an information system solution while following a disciplined software project life cycle approach. By terms end, your team must provide a sustainable solution that fits the clients objectives, organization constraints and capabilities"}'),
 Document(id='ef0e4b3a-61a4-4cd4-8f39-c18ae3d86a41', metadata={'source': '70-453', 'seq_num': 1, 'departmentid': '70', 'category': 'Courses'}, page_content='{"co_reqs": [], "anti_reqs": [], "equiv": [], "long_desc": "In this course, you will learn to how to effectively lead and undertake information system analysis and design projects. I

In [72]:
#print(retriever_major.invoke("MachineLearning"))
retriever_courses.invoke("Introduction to Multimedia Design")



[Document(id='866097c0-7c00-4378-8934-1263054d56b4', metadata={'source': '76-881', 'seq_num': 1, 'departmentid': '76', 'category': 'Courses'}, page_content='{"code": "76-881", "name": "Introduction to Multimedia Design", "base_name": "Introduction to Multimedia Design", "units": 12, "min_units": 12, "max_units": 12, "short_name": "MULTIMEDIA DESIGN", "is_topic": false, "prereqs": {"text": "(76-391 [] at least D) or (51-262 [] at least D) or (76-791 [] at least D)", "req_obj": {"id": 7912699, "screen_name": "ivzt7epm7wpnwaq", "original_min_units": null, "min_units": null, "is_shared": false, "is_uni_req": false, "is_concentration": false, "default_concentration": false, "choices": [{"id": 7912700, "screen_name": "msv9k494ryrnabm", "original_min_units": null, "min_units": null, "is_shared": false, "is_uni_req": false, "is_concentration": false, "default_concentration": false, "choices": [{"id": 511, "screen_name": "76-391", "constraints": [{"type": "course", "type_string": "", "data": {"

In [43]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableMap
from langchain.schema.runnable import RunnableLambda
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)


chain = (
    RunnableMap({
        "context": RunnableLambda(lambda x: retriever_courses.invoke(x)),
        "question": RunnablePassthrough()
    })
    | prompt
    | model
    | StrOutputParser()
)
''''chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)'''

'\'chain = (\n    {"context": retriever, "question": RunnablePassthrough()}\n    | prompt\n    | model\n    | StrOutputParser()\n)'

In [49]:
query = "76-530"
print(chain.invoke(query))

The context provided does not contain information about a course with the code "76-530".
